# Classification of hand-written digits with neural networks

In [ ]:
import torch  # PyTorch machine learning library
import torch.nn as nn  # provides a set of components for building neural networks
import torchvision  # provides computer-vision datasets
import torchvision.transforms as transforms  # provides a set of functions for transforming images
import matplotlib.pyplot as plt  # plotting tools

In [ ]:
# Check if a CUDA-capable GPU is available for acceleration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

**CUDA** is a parallel computing platform and programming model developed by Nvidia that enables developers to use NVIDIA GPUs for general-purpose computing. 
It allows software to offload computationally intensive and parallelizable tasks from the CPU to the GPU, leveraging the thousands of cores on modern graphics cards to achieve significant speedups.

## Loading the MNIST dataset of handwritten digits

The MNIST handwritten digit dataset is a widely used benchmark dataset in machine learning, consisting of 70,000 grayscale images of handwritten digits.
Each image is 28$\times$28 pixels in size, with pixel values ranging from 0 (black) to 255 (white), representing grayscale intensity.

The dataset is divided into a training set of 60,000 images and a test set of 10,000 images, with each digit class (0 to 9) having 7,000 examples distributed evenly across both sets.

In [ ]:
data_train = torchvision.datasets.MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
data_test = torchvision.datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor())

print(f'Number of Training examples: {len(data_train)}')
print(f'Number of Test examples: {len(data_test)}')

### Preparing the datasets

In [ ]:
# Number of examples in each training batch
batch_size_train = 100  #<<<<<

loader_train = torch.utils.data.DataLoader(dataset=data_train, batch_size=batch_size_train, shuffle=True)

num_batches = len(loader_train)
print(f'Number of training batches: {num_batches}')

# Number of examples in each test batch
batch_size_test = 50

loader_test = torch.utils.data.DataLoader(dataset=data_test, batch_size=batch_size_test, shuffle=True)
print(f'Number of test batches: {len(loader_test)}')

In [ ]:
# Checking a training batch
example_data, example_labels = next(iter(loader_train))

print(f'Shape of a training batch: {example_data.shape}')
print(f'Shape of the corresponding labels: {example_labels.shape}')

### Visualizing training examples

In [ ]:
for i in range(12):
    plt.subplot(3, 4, i+1)
    plt.imshow(example_data[i][0], cmap='gray')
    plt.title(f'Label: {example_labels[i]}')
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()

## Constructing the neural network (NN)

In [ ]:
class NN(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):
        super(NN, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size) # 1st hidden layer (linear)
        self.act1 = nn.ReLU() # 1st activation function, other options: Tanh(), ...
        self.linear2 = nn.Linear(hidden_size, output_size) # 2nd hidden layer (linear)

    def forward(self, x):
        a = self.act1(self.linear1(x))
        a = self.linear2(a)
        return a

### Defining hyperparametrs

In [ ]:
input_size = 28*28     # Size of input (i.e., dimension of images)
num_classes = 10       # Size of output (i.e., number of image classes)
hidden_size = 128      # Number of neurons in each hidden layer of the network  #<<<<<
num_epochs = 5         # Number of training iterations (epochs)  #<<<<<
learning_rate = 0.01   # Learning rate of the optimizer  #<<<<<

### Defining loss function and optimizer

In [ ]:
# Initializing a neural network using the NN class
model = NN(input_size, hidden_size, num_classes)

# Choosing a loss (cost) function:
# Cross-entropy loss function is particularly effective for classification tasks 
# because it provides strong gradients when predictions are incorrect, leading to 
# faster convergence.
loss_fn = nn.CrossEntropyLoss()

# Choosing an optimizer for training the network
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # available options: momentum=0.9  #<<<<<
#optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  #<<<<<

## Training the network

In [ ]:
# Create lists to store the training progress
train_losses, train_counter = [], []

# Reset the model parameters
for layer in model.children():
    if hasattr(layer, 'reset_parameters'):
        layer.reset_parameters()

# Train the network
for epoch in range(num_epochs):
    for i, (x_batch, y_batch) in enumerate(loader_train):

        # Initial shape of x_batch is [batch_size_train, 1, 28, 28], here 1 represents 1 color channel.
        # Reshaping it to [batch_size_train, 28, 28]
        x_batch = x_batch.reshape(-1, 28*28).to(device)
        y_batch = y_batch.to(device)

        # Forward pass (make prediction)
        y_pred = model(x_batch)

        # Compute the loss
        loss = loss_fn(y_pred, y_batch)

        # Backward pass (compute gradient of the loss)
        optimizer.zero_grad() # Reset the gradient to zero before computing the new gradient
        loss.backward() # Backpropagation step

        # Update network parameters based on the computed gradient
        optimizer.step()

        # Store and print the loss values each 50 steps
        if (i+1)%50 == 0:
            train_losses.append(loss.item())
            train_counter.append((i+1) + (epoch * num_batches))
            print(f'Epoch: {epoch+1:>2d}/{num_epochs}  \033[34m Batch: {i+1:>4d}/{num_batches} \033[0m \033[91m Loss = {loss.item():.4f} \033[0m')

print('Done!')

### Plotting the learning curve

In [ ]:
fig = plt.figure()
plt.plot(train_counter, train_losses, color='blue')
plt.legend(['training loss'], loc='upper right')
plt.xlabel('Number of training batches', labelpad=10)
plt.ylabel('Loss', labelpad=10)
plt.show()

## Testing the model

In [ ]:
with torch.no_grad():
    n_correct, n_samples = 0, 0
    for images, labels in loader_test:
        images = images.reshape(-1, 28*28)
        output = model(images) # Make prediction

        pred = torch.max(output, 1)[1] # Get the class labels with maximum probability (i.e., predicted classes)
        n_samples += labels.shape[0] # Add the total number of test samples in each batch
        n_correct += (pred == labels).sum().item() # Add the number of correctly classified samples in each batch
	
    accuracy = 100.0 * n_correct/n_samples
    print(f'accuracy = {accuracy}%')

In [ ]:
# Selecting a test batch
test_examples, _ = next(iter(loader_test))

with torch.no_grad():
    prediction = model(test_examples.reshape(-1, 28*28))

for i in range(12):
    plt.subplot(3, 4, i+1)
    plt.imshow(test_examples[i][0], cmap='gray')
    plt.title(f'Prediction: {torch.max(prediction, 1)[1][i]}')
    plt.xticks([])
    plt.yticks([])
plt.tight_layout()
plt.show()

<details>
<summary>Click here!</summary>
🎉

Congratulations! You successfully trained a neural network to classify handwritten digits.